In [1]:
"""
NetworkX Efficiency Benchmark — 10 Experiments
================================================
Run with:  python3 networkx_benchmark.py

Each experiment prints:
  - What graph is being tested (size + density)
  - Runtime in seconds
  - Peak memory in MB
  - A short interpretation of the result

No external downloads required — all graphs are generated synthetically.
"""

import time
import tracemalloc
import random
import networkx as nx
from networkx.generators.community import LFR_benchmark_graph
from networkx.algorithms.community import greedy_modularity_communities

# ─────────────────────────────────────────────
# SHARED UTILITIES
# ─────────────────────────────────────────────

DIVIDER = "─" * 60

def header(n, title):
    print(f"\n{'═' * 60}")
    print(f"  EXPERIMENT {n}: {title}")
    print(f"{'═' * 60}")

def run(label, func):
    """Time a function call and report runtime + peak memory."""
    tracemalloc.start()
    t0 = time.perf_counter()
    result = func()
    elapsed = time.perf_counter() - t0
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    print(f"  {label:<42} {elapsed:>7.3f}s   {peak/1e6:>7.1f} MB")
    return result

def graph_info(label, G):
    d = nx.density(G)
    print(f"  {DIVIDER}")
    print(f"  Graph : {label}")
    print(f"  Nodes : {G.number_of_nodes():,}   Edges: {G.number_of_edges():,}   Density: {d:.5f}")
    print(f"  {DIVIDER}")
    print(f"  {'Task':<42} {'Runtime':>9}   {'Peak RAM':>9}")
    print(f"  {DIVIDER}")

def make_er_graphs():
    """Return sparse, medium, and dense Erdős–Rényi graphs (1 000 nodes each)."""
    return {
        "Sparse  (p=0.01, ~5K edges)":  nx.erdos_renyi_graph(1000, 0.01,  seed=1),
        "Medium  (p=0.05, ~25K edges)": nx.erdos_renyi_graph(1000, 0.05,  seed=2),
        "Dense   (p=0.20, ~100K edges)":nx.erdos_renyi_graph(1000, 0.20,  seed=3),
    }

# ─────────────────────────────────────────────
# EXPERIMENT 1 — GRAPH CONSTRUCTION SPEED
# ─────────────────────────────────────────────
def exp1():
    header(1, "Graph Construction Speed")
    print("""
  WHAT IT DOES:
    Builds graphs of increasing size from an edge list in memory.
    This isolates the cost of NetworkX's internal Python-dict graph
    representation — every node and edge is a Python object.

  WHAT TO LOOK FOR:
    Runtime and memory should grow roughly linearly with edge count.
    Even at moderate sizes (100K+ edges) you will see meaningful
    slowdowns compared to C-based libraries, because each add_edge()
    call touches Python dicts for both endpoints.
""")
    print(f"  {'Task':<42} {'Runtime':>9}   {'Peak RAM':>9}")
    print(f"  {DIVIDER}")

    configs = [
        ("G(n=500,  p=0.05)  ~6K edges",   500,  0.05),
        ("G(n=1000, p=0.05)  ~25K edges",  1000, 0.05),
        ("G(n=2000, p=0.05)  ~100K edges", 2000, 0.05),
        ("G(n=3000, p=0.05)  ~225K edges", 3000, 0.05),
    ]
    for label, n, p in configs:
        edges = [(u, v) for u in range(n)
                        for v in range(u + 1, n)
                        if random.random() < p]
        run(label, lambda e=edges: nx.Graph(e))

    print("\n  INTERPRETATION:")
    print("  Memory grows quickly — each node/edge stores Python dicts.")
    print("  Compare MB values: even 225K edges can cost 100+ MB of RAM.")

# ─────────────────────────────────────────────
# EXPERIMENT 2 — SINGLE-SOURCE SHORTEST PATH
# ─────────────────────────────────────────────
def exp2():
    header(2, "Single-Source Shortest Path (BFS)")
    print("""
  WHAT IT DOES:
    Runs BFS from one source node to find the shortest hop-count
    distance to every other reachable node.  Uses
    nx.single_source_shortest_path_length().

  WHAT TO LOOK FOR:
    Runtime climbs steeply as density increases — a denser graph
    means BFS must visit far more edges at each layer.
    On a truly large dense graph (Pokec social network, 30M edges)
    this takes ~68 seconds in NetworkX vs <1 second in Networkit.
""")
    graphs = make_er_graphs()
    for label, G in graphs.items():
        graph_info(label, G)
        src = list(G.nodes())[0]
        run("SSSP from node 0", lambda G=G, s=src:
            dict(nx.single_source_shortest_path_length(G, s)))

    print("\n  INTERPRETATION:")
    print("  Watch the Dense row — runtime can be 10-50× the Sparse row.")
    print("  The gap widens further as n grows.")

# ─────────────────────────────────────────────
# EXPERIMENT 3 — BETWEENNESS CENTRALITY
# ─────────────────────────────────────────────
def exp3():
    header(3, "Betweenness Centrality — Exact vs. Approximate")
    print("""
  WHAT IT DOES:
    Betweenness centrality scores each node by how many shortest
    paths pass through it.  Exact computation is O(n × m) — one
    full BFS/Dijkstra pass per node.  The k parameter samples only
    k pivot nodes instead of all n, giving an approximation.

  WHAT TO LOOK FOR:
    - Exact (k=None): runtime explodes on dense graphs.
    - Increasing k: runtime rises but accuracy improves.
    - Small k (k=10) is fast but can miss important nodes.
    - k=n is equivalent to exact.
    On a dense graph with 1 000 nodes, exact can take 30+ seconds.
""")
    G_sparse = nx.erdos_renyi_graph(600, 0.02, seed=10)
    G_dense  = nx.erdos_renyi_graph(600, 0.15, seed=11)

    for label, G in [("Sparse (p=0.02)", G_sparse), ("Dense (p=0.15)", G_dense)]:
        graph_info(label, G)
        run("Exact  (k=None)",  lambda G=G: nx.betweenness_centrality(G))
        for k in [10, 50, 100]:
            run(f"Approx (k={k})",
                lambda G=G, k=k: nx.betweenness_centrality(G, k=k, seed=42))

    print("\n  INTERPRETATION:")
    print("  Exact on Dense will be dramatically slower than Sparse.")
    print("  k=50 usually gives a good accuracy/speed trade-off.")

# ─────────────────────────────────────────────
# EXPERIMENT 4 — PAGERANK CONVERGENCE
# ─────────────────────────────────────────────
def exp4():
    header(4, "PageRank — Damping Factor & Density")
    print("""
  WHAT IT DOES:
    PageRank iteratively distributes 'importance' across nodes
    via their edges.  The damping factor alpha controls how much
    weight flows along edges vs. teleporting randomly.
    Higher alpha = more iterations needed to converge.

  WHAT TO LOOK FOR:
    - alpha=0.50: converges fast (fewer iterations).
    - alpha=0.99: many more iterations, much slower.
    - Dense graphs converge slower because each node has more
      neighbors pulling its score in different directions.
    - Memory stays moderate — PageRank stores one float per node.
""")
    G = nx.erdos_renyi_graph(2000, 0.05, seed=20)
    graph_info(f"Fixed graph (n=2000, p=0.05)", G)
    for alpha in [0.50, 0.75, 0.85, 0.95, 0.99]:
        run(f"PageRank alpha={alpha}",
            lambda a=alpha: nx.pagerank(G, alpha=a, max_iter=200))

    print()
    G2 = nx.erdos_renyi_graph(1500, 0.01, seed=21)
    G3 = nx.erdos_renyi_graph(1500, 0.10, seed=22)
    print("  Density comparison (alpha=0.85 fixed):")
    print(f"  {'Task':<42} {'Runtime':>9}   {'Peak RAM':>9}")
    print(f"  {DIVIDER}")
    run(f"Sparse (p=0.01, density={nx.density(G2):.4f})",
        lambda: nx.pagerank(G2, alpha=0.85))
    run(f"Dense  (p=0.10, density={nx.density(G3):.4f})",
        lambda: nx.pagerank(G3, alpha=0.85))

    print("\n  INTERPRETATION:")
    print("  High alpha + dense graph = slowest combination.")
    print("  If PageRank fails to converge, increase max_iter.")

# ─────────────────────────────────────────────
# EXPERIMENT 5 — COMMUNITY DETECTION
# ─────────────────────────────────────────────
def exp5():
    header(5, "Community Detection — LFR mu parameter")
    print("""
  WHAT IT DOES:
    Uses greedy_modularity_communities() to find clusters.
    The LFR benchmark generates graphs with planted communities.
    mu controls the fraction of inter-community edges:
      mu=0.1 → tight clusters, easy to find
      mu=0.4 → blurry clusters, algorithm must work harder

  WHAT TO LOOK FOR:
    - Runtime rises as mu increases: more inter-community edges
      means more candidate merges to evaluate at each step.
    - The number of detected communities may also change —
      high mu causes communities to bleed into each other.
    - Compare detected_communities to the true planted count.
""")
    print(f"  {'Task':<42} {'Runtime':>9}   {'Detected communities':>22}")
    print(f"  {DIVIDER}")

    for mu in [0.1, 0.2, 0.3, 0.4]:
        try:
            G_lfr = LFR_benchmark_graph(
                400, tau1=3, tau2=1.5, mu=mu,
                average_degree=8, min_community=20, seed=42
            )
            true_communities = len({frozenset(G_lfr.nodes[v]['community'])
                                    for v in G_lfr})
            tracemalloc.start()
            t0 = time.perf_counter()
            comms = list(greedy_modularity_communities(G_lfr))
            elapsed = time.perf_counter() - t0
            _, peak = tracemalloc.get_traced_memory()
            tracemalloc.stop()
            print(f"  mu={mu}  (true={true_communities} communities)"
                  f"{'':>10}{elapsed:>7.3f}s   "
                  f"{peak/1e6:>6.1f} MB   detected={len(comms)}")
        except Exception as e:
            print(f"  mu={mu}: skipped ({e})")

    print("\n  INTERPRETATION:")
    print("  High mu = blurry structure. Detected count drifts from true count.")
    print("  Runtime increase shows the algorithm struggling with ambiguity.")

# ─────────────────────────────────────────────
# EXPERIMENT 6 — K-CORE DECOMPOSITION
# ─────────────────────────────────────────────
def exp6():
    header(6, "K-Core Decomposition")
    print("""
  WHAT IT DOES:
    The k-core of a graph is the maximal subgraph where every node
    has at least k neighbours within that subgraph.  nx.core_number()
    assigns each node its highest qualifying k.  This is used to find
    the 'densest heart' of a network.

  WHAT TO LOOK FOR:
    - Runtime scales with edge count — more edges = more pruning rounds.
    - Dense graphs have high maximum core numbers (large k-cores).
    - Sparse graphs may have a maximum core of just 2 or 3.
    - This is one of NetworkX's most severe bottlenecks:
      Networkit is ~2000× faster on large graphs.
    - Print the max core number to understand graph structure.
""")
    graphs = make_er_graphs()
    for label, G in graphs.items():
        graph_info(label, G)
        cores = run("core_number(G)", lambda G=G: nx.core_number(G))
        if cores:
            print(f"  → Max core number: {max(cores.values())}")

    print("\n  INTERPRETATION:")
    print("  Dense graph will have a much higher max core number.")
    print("  That also means more work for the peeling algorithm.")

# ─────────────────────────────────────────────
# EXPERIMENT 7 — CLUSTERING COEFFICIENT
# ─────────────────────────────────────────────
def exp7():
    header(7, "Clustering Coefficient")
    print("""
  WHAT IT DOES:
    The clustering coefficient of a node measures the fraction of
    its neighbours that are also connected to each other (triangles).
    nx.average_clustering() averages this over all nodes.
    nx.transitivity() computes the global ratio of closed triangles.

  WHAT TO LOOK FOR:
    - Transitivity is cheaper than average_clustering because it
      counts triangles globally rather than per-node.
    - Dense graphs have far more triangles → much slower.
    - The coefficient value itself rises with density.
    - Watts-Strogatz small-world graphs have high clustering
      even at moderate density — good comparison point.
""")
    G_sparse = nx.erdos_renyi_graph(1500, 0.02, seed=30)
    G_dense  = nx.erdos_renyi_graph(1500, 0.12, seed=31)
    G_sw     = nx.watts_strogatz_graph(1500, 12, 0.1, seed=32)

    for label, G in [
        ("Sparse ER   (p=0.02)", G_sparse),
        ("Dense  ER   (p=0.12)", G_dense),
        ("Small-World (k=12)",   G_sw),
    ]:
        graph_info(label, G)
        run("transitivity(G)       [global]",
            lambda G=G: nx.transitivity(G))
        result = run("average_clustering(G)  [per-node]",
                     lambda G=G: nx.average_clustering(G))
        print(f"  → Coefficient value: {result:.4f}")

    print("\n  INTERPRETATION:")
    print("  Dense graph: high coefficient, high runtime — both grow together.")
    print("  Small-world: high clustering but moderate runtime (sparse edges).")

# ─────────────────────────────────────────────
# EXPERIMENT 8 — CONNECTED COMPONENTS
# ─────────────────────────────────────────────
def exp8():
    header(8, "Connected Components")
    print("""
  WHAT IT DOES:
    nx.connected_components() finds all disconnected subgraphs using
    BFS/DFS from each unvisited node.  It is one of the few NetworkX
    algorithms that performs comparably to C-based alternatives.

  WHAT TO LOOK FOR:
    - Very sparse graphs (p << 1/n) have many small components.
    - As p rises past ~ln(n)/n the graph becomes fully connected
      (one giant component).  For n=2000 that threshold is ~0.004.
    - Runtime stays fast even on larger graphs — this is a
      best-case experiment showing where NetworkX is competitive.
    - Track number_of_components: it should drop sharply past threshold.
""")
    n = 2000
    threshold = 0.004   # ln(2000)/2000 ≈ 0.0038
    print(f"  Giant-component threshold for n={n}: p ≈ {threshold:.4f}\n")
    print(f"  {'Task':<42} {'Runtime':>9}   {'Components':>12}")
    print(f"  {DIVIDER}")

    for p in [0.001, 0.003, 0.004, 0.006, 0.01, 0.05]:
        G = nx.erdos_renyi_graph(n, p, seed=40)
        tracemalloc.start()
        t0 = time.perf_counter()
        comps = list(nx.connected_components(G))
        elapsed = time.perf_counter() - t0
        _, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        giant = max(len(c) for c in comps)
        print(f"  p={p:<6}  {n} nodes"
              f"{'':>12}{elapsed:>7.3f}s   "
              f"{peak/1e6:>5.1f} MB   "
              f"components={len(comps):>5}  giant={giant}")

    print("\n  INTERPRETATION:")
    print("  Components drops from thousands to 1 right around the threshold.")
    print("  Runtime stays flat — this is NetworkX at its most efficient.")

# ─────────────────────────────────────────────
# EXPERIMENT 9 — MEMORY FOOTPRINT SCALING
# ─────────────────────────────────────────────
def exp9():
    header(9, "Memory Footprint vs. Graph Size")
    print("""
  WHAT IT DOES:
    Builds graphs of increasing node count and measures peak RAM
    using tracemalloc.  No algorithm is run — just graph creation.
    This isolates the memory cost of NetworkX's adjacency-dict
    representation (every node maps to a dict of its neighbours).

  WHAT TO LOOK FOR:
    - Memory grows super-linearly with edges (not just nodes).
    - A graph with 50K nodes and p=0.05 has ~62M edges — NetworkX
      stores each as a Python dict entry (≈200-400 bytes each).
    - Very large dense graphs will hit out-of-memory before an
      algorithm can even run. This is NetworkX's hard ceiling.
    - Compare MB/edge across rows — should stay roughly constant,
      showing the per-edge overhead of Python objects.
""")
    print(f"  {'Graph':<35} {'Nodes':>8} {'Edges':>10} {'Peak RAM':>10} {'MB/edge':>9}")
    print(f"  {DIVIDER}")

    configs = [
        (500,   0.05),
        (1000,  0.05),
        (2000,  0.05),
        (4000,  0.05),
        (1000,  0.20),   # same nodes, much denser
        (1000,  0.50),
    ]

    for n, p in configs:
        tracemalloc.start()
        G = nx.erdos_renyi_graph(n, p, seed=50)
        _, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        e = G.number_of_edges()
        mb_per_edge = (peak / 1e6 / e) if e > 0 else 0
        print(f"  n={n:<5} p={p:<5}"
              f"{n:>10,}{e:>10,}{peak/1e6:>10.1f} MB{mb_per_edge:>9.4f}")

    print("\n  INTERPRETATION:")
    print("  MB/edge stays roughly constant — each edge costs ~same RAM.")
    print("  Density rows show: same n but 10× more RAM with 10× more edges.")

# ─────────────────────────────────────────────
# EXPERIMENT 10 — DYNAMIC GRAPH MODIFICATION
# ─────────────────────────────────────────────
def exp10():
    header(10, "Dynamic Graph Modification")
    print("""
  WHAT IT DOES:
    Measures the cost of live graph edits: adding nodes, adding edges,
    removing edges, and removing nodes (which cascades to their edges).
    These are common in streaming data scenarios.

  WHAT TO LOOK FOR:
    - add_edges_from() is batched and faster than one-at-a-time adds.
    - remove_nodes_from() costs more per item than remove_edges_from()
      because each node removal also deletes all its edges.
    - Denser starting graphs cost more to modify because neighbor
      dicts are larger and take longer to update.
    - If you are building a graph incrementally, prefer batch methods
      (add_edges_from) over calling add_edge() in a loop.
""")
    random.seed(60)

    for label, p in [("Sparse base graph (p=0.02)", 0.02),
                     ("Dense  base graph (p=0.15)", 0.15)]:
        G = nx.erdos_renyi_graph(2000, p, seed=61)
        nodes = list(G.nodes())
        graph_info(label, G)

        # Add 5 000 new edges
        new_edges = [(random.choice(nodes), random.choice(nodes))
                     for _ in range(5000)]
        run("add_edges_from() — 5 000 edges",
            lambda G=G, e=new_edges: G.add_edges_from(e))

        # Add edges one at a time (slower pattern)
        run("add_edge() loop    — 500 edges",
            lambda G=G, ns=nodes: [G.add_edge(random.choice(ns),
                                              random.choice(ns))
                                   for _ in range(500)])

        # Remove 1 000 existing edges
        existing = list(G.edges())[:1000]
        run("remove_edges_from()— 1 000 edges",
            lambda G=G, e=existing: G.remove_edges_from(e))

        # Remove 100 nodes (cascades to their edges)
        to_remove = nodes[:100]
        run("remove_nodes_from()— 100 nodes",
            lambda G=G, ns=to_remove: G.remove_nodes_from(ns))

    print("\n  INTERPRETATION:")
    print("  Batch add_edges_from >> repeated add_edge() — always prefer batch.")
    print("  Dense graph modifications cost more due to larger neighbour dicts.")
    print("  Node removal is expensive — it must clean up every incident edge.")

# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────
if __name__ == "__main__":
    random.seed(0)

    print("\n" + "═" * 60)
    print("  NetworkX Efficiency Benchmark — 10 Experiments")
    print(f"  NetworkX version: {nx.__version__}")
    print("═" * 60)
    print("  Columns: Task | Runtime (seconds) | Peak RAM (MB)")
    print("  All graphs generated synthetically — no downloads needed.")

    exp1()
    exp2()
    exp3()
    exp4()
    exp5()
    exp6()
    exp7()
    exp8()
    exp9()
    exp10()

    print("\n" + "═" * 60)
    print("  All 10 experiments complete.")
    print("═" * 60 + "\n")


════════════════════════════════════════════════════════════
  NetworkX Efficiency Benchmark — 10 Experiments
  NetworkX version: 3.7rc0.dev0
════════════════════════════════════════════════════════════
  Columns: Task | Runtime (seconds) | Peak RAM (MB)
  All graphs generated synthetically — no downloads needed.

════════════════════════════════════════════════════════════
  EXPERIMENT 1: Graph Construction Speed
════════════════════════════════════════════════════════════

  WHAT IT DOES:
    Builds graphs of increasing size from an edge list in memory.
    This isolates the cost of NetworkX's internal Python-dict graph
    representation — every node and edge is a Python object.

  WHAT TO LOOK FOR:
    Runtime and memory should grow roughly linearly with edge count.
    Even at moderate sizes (100K+ edges) you will see meaningful
    slowdowns compared to C-based libraries, because each add_edge()
    call touches Python dicts for both endpoints.

  Task                           